# CMIP6 Heat Story Data Builder: Average Warming → Extreme Hot Days

This Colab notebook prepares CSV files for the DSC 106 final project direction:

**Average warming is abstract, but extreme hot days change daily life.**

It keeps the original annual state/county outputs that your current D3 map already expects, and adds new **state-level summer and monthly outputs** for the revised story.

## Main outputs for the website

Annual files compatible with your current `main.js`:

1. `state_annual_tas_cmip6_2020_2070.csv`
2. `state_annual_extreme_heat_cmip6_2020_2070.csv`
3. `county_annual_tas_cmip6_2020_2070.csv`
4. `county_annual_extreme_heat_cmip6_2020_2070.csv`
5. `us_states.geojson`
6. `us_counties.geojson`

New files for the updated story:

7. `state_summer_heat_life_cmip6_2020_2070.csv`  
   Summer-only average temperature and hot-day counts.

8. `state_monthly_heat_life_cmip6_2020_2070.csv`  
   Monthly average temperature and hot-day counts for selected-state detail charts.

9. `state_annual_heat_life_combined_cmip6_2020_2070.csv`  
   One combined annual state file with average temperature + hot days.

10. `state_summer_heat_life_combined_cmip6_2020_2070.csv`  
   One combined summer state file with average temperature + hot days.

11. `state_monthly_heat_life_combined_cmip6_2020_2070.csv`  
   One combined monthly state file with average temperature + hot days.

## Important interpretation note

County-level values are estimated by sampling the nearest CMIP6 grid cell at each county's representative point. They are useful for exploratory spatial comparison, not precise county-scale prediction.

For the final website, the **state-level outputs** are the safest and fastest to use.

## 0. Install packages

Run this first in Google Colab.

In [ ]:
%%capture
!pip install geopandas pyogrio shapely rtree fsspec gcsfs zarr xarray dask[complete] netCDF4 cftime intake-esm tqdm

## 1. Imports and configuration

Recommended setup for the final project:

- Average temperature: `tas`
- Extreme hot days: daily `tasmax`
- Annual + summer + monthly aggregations
- Summer months: June, July, August
- Scenarios: SSP126, SSP245, SSP585
- Model: GFDL-ESM4

In [ ]:
import os
import gc
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import intake
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
xr.set_options(display_style="text")

OUTPUT_DIR = Path("/content/cmip6_heat_story_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------
# Time setup
# -----------------------
BASELINE_YEAR = 2020
START_YEAR = 2020
END_YEAR = 2070
ALL_YEARS = list(range(START_YEAR, END_YEAR + 1))

# -----------------------
# Seasonal setup
# -----------------------
SUMMER_MONTHS = [6, 7, 8]
SUMMER_LABEL = "JJA"

# -----------------------
# CMIP6 setup
# -----------------------
SOURCE_ID = "GFDL-ESM4"
MEMBER_ID = "r1i1p1f1"
SCENARIOS = ["ssp126", "ssp245", "ssp585"]
SCENARIO_LABELS = {
    "ssp126": "Low emissions (SSP126)",
    "ssp245": "Medium emissions (SSP245)",
    "ssp585": "High emissions (SSP585)",
}

# Extreme heat thresholds in Celsius.
# 35°C is more intuitive as "extreme heat"; 30°C is useful if 35°C is too sparse in northern states.
HOT_THRESHOLDS_C = [30, 35]

# Census cartographic boundary files.
COUNTY_ZIP_URL = "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_county_20m.zip"
STATE_ZIP_URL = "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_state_20m.zip"

# Keep 50 states only. Exclude DC and territories.
STATE_ABBRS_50 = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA","KS",
    "KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY",
    "NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV",
    "WI","WY"
}

print("Output directory:", OUTPUT_DIR)
print("Years:", START_YEAR, "to", END_YEAR)
print("Scenarios:", SCENARIOS)
print("Summer months:", SUMMER_MONTHS)
print("Hot thresholds:", HOT_THRESHOLDS_C)

## 2. Load state and county boundaries

This creates county representative points for sampling CMIP6 grid cells.

In [ ]:
states = gpd.read_file(STATE_ZIP_URL).to_crs("EPSG:4326")
counties = gpd.read_file(COUNTY_ZIP_URL).to_crs("EPSG:4326")

# Keep only 50 states.
states = states[states["STUSPS"].isin(STATE_ABBRS_50)].copy()

# Join state names/abbreviations onto counties.
state_lookup = states[["STATEFP", "NAME", "STUSPS"]].rename(
    columns={"NAME": "state", "STUSPS": "state_abbr"}
)

counties = counties.merge(state_lookup, on="STATEFP", how="inner")
counties = counties[counties["state_abbr"].isin(STATE_ABBRS_50)].copy()
counties = counties.rename(columns={"NAME": "county"})

# Make sure key columns are strings.
counties["GEOID"] = counties["GEOID"].astype(str).str.zfill(5)
counties["COUNTYFP"] = counties["COUNTYFP"].astype(str).str.zfill(3)
counties["STATEFP"] = counties["STATEFP"].astype(str).str.zfill(2)

county_meta = counties[
    ["GEOID", "STATEFP", "COUNTYFP", "county", "state", "state_abbr", "ALAND", "geometry"]
].copy()

# Representative point is safer than centroid because it stays inside the polygon.
rep_points = county_meta.geometry.representative_point()
county_meta["rep_lon"] = rep_points.x
county_meta["rep_lat"] = rep_points.y

county_attrs = county_meta[["GEOID", "county", "state", "state_abbr", "ALAND"]].copy()

print("States:", len(states))
print("Counties:", len(county_meta))
display(county_meta[["GEOID", "county", "state", "state_abbr", "rep_lon", "rep_lat"]].head())

## 3. Open the CMIP6 catalog and check availability

In [ ]:
CATALOG_URL = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
cat = intake.open_esm_datastore(CATALOG_URL)

print("Catalog rows:", len(cat.df))
display(cat.df.head())

In [ ]:
# Check whether the needed datasets exist.
REQUIRED_DATASETS = [
    ("tas", "Amon"),
    ("tasmax", "day"),
]

availability_rows = []

for scenario in SCENARIOS:
    for variable_id, table_id in REQUIRED_DATASETS:
        search = cat.search(
            source_id=SOURCE_ID,
            experiment_id=scenario,
            table_id=table_id,
            variable_id=variable_id,
            member_id=MEMBER_ID,
        )
        availability_rows.append({
            "scenario": scenario,
            "variable_id": variable_id,
            "table_id": table_id,
            "matches": len(search.df),
        })

availability = pd.DataFrame(availability_rows)
display(availability)

if (availability["matches"] == 0).any():
    raise ValueError("At least one required CMIP6 dataset is missing. Check SOURCE_ID, MEMBER_ID, scenarios, or variables.")

## 4. Helper functions

In [ ]:
def _get_coord_name(ds, possible_names):
    for name in possible_names:
        if name in ds.coords or name in ds.dims:
            return name
    raise ValueError(
        f"Could not find coordinate among {possible_names}. "
        f"Available coords: {list(ds.coords)}"
    )


def open_cmip6_variable(scenario, variable_id, table_id):
    # Open one CMIP6 variable for one scenario and subset to 2020-2070.
    search = cat.search(
        source_id=SOURCE_ID,
        experiment_id=scenario,
        table_id=table_id,
        variable_id=variable_id,
        member_id=MEMBER_ID,
    )

    if len(search.df) == 0:
        raise ValueError(
            f"No CMIP6 catalog entries found for "
            f"scenario={scenario}, variable={variable_id}, table={table_id}"
        )

    print(
        f"Found {len(search.df)} catalog entries for "
        f"{scenario}, {variable_id}, {table_id}. Using first matching dataset."
    )

    dsets = search.to_dataset_dict(
        zarr_kwargs={"consolidated": True, "use_cftime": True},
        storage_options={"token": "anon"},
        progressbar=True,
    )

    ds = list(dsets.values())[0]

    lat_name = _get_coord_name(ds, ["lat", "latitude", "y"])
    lon_name = _get_coord_name(ds, ["lon", "longitude", "x"])

    da = ds[variable_id]
    da = da.sel(time=slice(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"))

    return da, lat_name, lon_name


def to_celsius(da):
    # CMIP6 temperature variables are usually in Kelvin; convert to Celsius.
    return da - 273.15


def sample_county_timeseries(da, lat_name, lon_name, county_meta):
    # Sample an xarray DataArray at county representative points using nearest grid cell.
    lon_values = da[lon_name].values
    lon_min = np.nanmin(lon_values)
    lon_max = np.nanmax(lon_values)

    county_lons = county_meta["rep_lon"].to_numpy()

    # CMIP6 longitudes are often 0..360. Convert county lon if needed.
    if lon_min >= 0 and lon_max > 180:
        county_lons_for_model = (county_lons + 360) % 360
    else:
        county_lons_for_model = county_lons

    county_lats_da = xr.DataArray(
        county_meta["rep_lat"].to_numpy(),
        dims="county_index",
        coords={"county_index": county_meta["GEOID"].to_numpy()},
    )

    county_lons_da = xr.DataArray(
        county_lons_for_model,
        dims="county_index",
        coords={"county_index": county_meta["GEOID"].to_numpy()},
    )

    sampled = da.sel(
        {
            lat_name: county_lats_da,
            lon_name: county_lons_da,
        },
        method="nearest",
    )

    sampled = sampled.assign_coords(county_index=county_meta["GEOID"].to_numpy())
    return sampled


def annual_da_to_county_df(annual_da, value_name, scenario):
    # Convert annual xarray DataArray with dims time and county_index into a clean DataFrame.
    df = annual_da.to_dataframe(name=value_name).reset_index()
    keep_cols = [c for c in ["county_index", "time", value_name] if c in df.columns]
    df = df[keep_cols].copy()

    df["year"] = df["time"].apply(lambda t: t.year)
    df = df[df["year"].isin(ALL_YEARS)].copy()
    df = df.rename(columns={"county_index": "GEOID"})
    df["GEOID"] = df["GEOID"].astype(str).str.zfill(5)
    df["scenario"] = scenario
    df["scenario_label"] = SCENARIO_LABELS[scenario]

    return df[["GEOID", "year", "scenario", "scenario_label", value_name]]


def monthly_da_to_county_df(monthly_da, value_name, scenario):
    # Convert monthly xarray DataArray with dims time and county_index into a clean DataFrame.
    df = monthly_da.to_dataframe(name=value_name).reset_index()
    keep_cols = [c for c in ["county_index", "time", value_name] if c in df.columns]
    df = df[keep_cols].copy()

    df["year"] = df["time"].apply(lambda t: t.year)
    df["month"] = df["time"].apply(lambda t: t.month)
    df = df[df["year"].isin(ALL_YEARS)].copy()
    df = df.rename(columns={"county_index": "GEOID"})
    df["GEOID"] = df["GEOID"].astype(str).str.zfill(5)
    df["scenario"] = scenario
    df["scenario_label"] = SCENARIO_LABELS[scenario]

    return df[["GEOID", "year", "month", "scenario", "scenario_label", value_name]]


def weighted_average(group, value_col, weight_col="ALAND"):
    valid = (
        group[value_col].notna() &
        group[weight_col].notna() &
        (group[weight_col] > 0)
    )
    if valid.sum() == 0:
        return np.nan
    return np.average(group.loc[valid, value_col], weights=group.loc[valid, weight_col])


def aggregate_county_to_state(county_df, value_cols, extra_group_cols=None):
    # Area-weight county rows to state rows using county ALAND.
    if extra_group_cols is None:
        extra_group_cols = ["year"]

    group_cols = ["state", "state_abbr"] + extra_group_cols + ["scenario", "scenario_label"]
    state_records = []

    for keys, group in county_df.groupby(group_cols, dropna=False):
        record = dict(zip(group_cols, keys))
        for col in value_cols:
            record[col] = weighted_average(group, col)
        state_records.append(record)

    sort_cols = ["state", "scenario"] + extra_group_cols
    return pd.DataFrame(state_records).sort_values(sort_cols)


def add_annual_baseline_change(df, id_cols, value_col, baseline_year=BASELINE_YEAR):
    # Add baseline and change-from-baseline columns for annual/summer files.
    baseline = df[df["year"] == baseline_year][id_cols + [value_col]].copy()
    baseline = baseline.rename(columns={value_col: f"{value_col}_2020"})
    out = df.merge(baseline, on=id_cols, how="left")
    out[f"{value_col}_change_from_2020"] = out[value_col] - out[f"{value_col}_2020"]
    return out


def add_monthly_baseline_change(df, id_cols, value_col, baseline_year=BASELINE_YEAR):
    # Add baseline and change-from-baseline columns for monthly files.
    # Baseline is matched by the same calendar month in 2020.
    baseline_cols = id_cols + ["month"]
    baseline = df[df["year"] == baseline_year][baseline_cols + [value_col]].copy()
    baseline = baseline.rename(columns={value_col: f"{value_col}_2020_same_month"})
    out = df.merge(baseline, on=baseline_cols, how="left")
    out[f"{value_col}_change_from_2020_same_month"] = (
        out[value_col] - out[f"{value_col}_2020_same_month"]
    )
    return out

## 5. Build average temperature files from `tas`

This section uses `tas` from CMIP6 monthly data (`Amon`).

It creates:

- annual average temperature
- summer-only average temperature
- monthly average temperature

In [ ]:
county_tas_parts = []
county_tas_summer_parts = []
state_tas_monthly_parts = []

for scenario in SCENARIOS:
    print("\n==============================")
    print("Processing tas:", scenario)
    print("==============================")

    tas, lat_name, lon_name = open_cmip6_variable(
        scenario=scenario,
        variable_id="tas",
        table_id="Amon",
    )

    sampled_tas = sample_county_timeseries(tas, lat_name, lon_name, county_meta)
    sampled_tas_c = to_celsius(sampled_tas)
    sampled_tas_c.name = "tas_c"

    # -----------------------
    # Annual average tas
    # -----------------------
    annual_tas_c = sampled_tas_c.resample(time="YS").mean()
    county_scenario_tas = annual_da_to_county_df(
        annual_da=annual_tas_c,
        value_name="tas_c",
        scenario=scenario,
    )
    county_tas_parts.append(county_scenario_tas)

    # -----------------------
    # Summer average tas
    # -----------------------
    summer_tas_c = (
        sampled_tas_c
        .where(sampled_tas_c["time"].dt.month.isin(SUMMER_MONTHS), drop=True)
        .resample(time="YS")
        .mean()
    )
    county_scenario_summer_tas = annual_da_to_county_df(
        annual_da=summer_tas_c,
        value_name="summer_tas_c",
        scenario=scenario,
    )
    county_scenario_summer_tas["season"] = SUMMER_LABEL
    county_tas_summer_parts.append(county_scenario_summer_tas)

    # -----------------------
    # Monthly average tas
    # -----------------------
    # tas is already monthly, but resample to MS makes the timestamp consistent.
    monthly_tas_c = sampled_tas_c.resample(time="MS").mean()
    county_scenario_monthly_tas = monthly_da_to_county_df(
        monthly_da=monthly_tas_c,
        value_name="monthly_tas_c",
        scenario=scenario,
    )
    county_scenario_monthly_tas = county_scenario_monthly_tas.merge(county_attrs, on="GEOID", how="left")
    state_scenario_monthly_tas = aggregate_county_to_state(
        county_scenario_monthly_tas,
        value_cols=["monthly_tas_c"],
        extra_group_cols=["year", "month"],
    )
    state_tas_monthly_parts.append(state_scenario_monthly_tas)

    # Free memory before next scenario.
    del tas, sampled_tas, sampled_tas_c
    del annual_tas_c, county_scenario_tas
    del summer_tas_c, county_scenario_summer_tas
    del monthly_tas_c, county_scenario_monthly_tas, state_scenario_monthly_tas
    gc.collect()

# -----------------------
# Annual tas: county + state
# -----------------------
county_tas = pd.concat(county_tas_parts, ignore_index=True)
county_tas = county_tas.merge(county_attrs, on="GEOID", how="left")

county_tas = add_annual_baseline_change(
    county_tas,
    id_cols=["GEOID", "scenario"],
    value_col="tas_c",
)

county_tas = county_tas[
    [
        "GEOID", "county", "state", "state_abbr", "ALAND",
        "year", "scenario", "scenario_label",
        "tas_c", "tas_c_2020", "tas_c_change_from_2020",
    ]
].rename(columns={
    "tas_c_2020": "tas_2020_c",
    "tas_c_change_from_2020": "tas_change_from_2020_c",
}).sort_values(["state", "county", "scenario", "year"])

state_tas = aggregate_county_to_state(
    county_tas,
    value_cols=["tas_c", "tas_2020_c", "tas_change_from_2020_c"],
    extra_group_cols=["year"],
)

# -----------------------
# Summer tas: county + state
# -----------------------
county_summer_tas = pd.concat(county_tas_summer_parts, ignore_index=True)
county_summer_tas = county_summer_tas.merge(county_attrs, on="GEOID", how="left")

county_summer_tas = add_annual_baseline_change(
    county_summer_tas,
    id_cols=["GEOID", "scenario"],
    value_col="summer_tas_c",
)

county_summer_tas = county_summer_tas[
    [
        "GEOID", "county", "state", "state_abbr", "ALAND",
        "year", "season", "scenario", "scenario_label",
        "summer_tas_c", "summer_tas_c_2020", "summer_tas_c_change_from_2020",
    ]
].sort_values(["state", "county", "scenario", "year"])

state_summer_tas = aggregate_county_to_state(
    county_summer_tas,
    value_cols=["summer_tas_c", "summer_tas_c_2020", "summer_tas_c_change_from_2020"],
    extra_group_cols=["year", "season"],
)

# -----------------------
# Monthly tas: state only
# -----------------------
state_monthly_tas = pd.concat(state_tas_monthly_parts, ignore_index=True)
state_monthly_tas = add_monthly_baseline_change(
    state_monthly_tas,
    id_cols=["state", "state_abbr", "scenario"],
    value_col="monthly_tas_c",
)

print("county_tas rows:", len(county_tas))
print("state_tas rows:", len(state_tas))
print("state_summer_tas rows:", len(state_summer_tas))
print("state_monthly_tas rows:", len(state_monthly_tas))

display(state_tas.head())
display(state_summer_tas.head())
display(state_monthly_tas.head())

In [ ]:
county_tas_csv = OUTPUT_DIR / "county_annual_tas_cmip6_2020_2070.csv"
state_tas_csv = OUTPUT_DIR / "state_annual_tas_cmip6_2020_2070.csv"
state_summer_tas_csv = OUTPUT_DIR / "state_summer_tas_cmip6_2020_2070.csv"
state_monthly_tas_csv = OUTPUT_DIR / "state_monthly_tas_cmip6_2020_2070.csv"

county_tas.to_csv(county_tas_csv, index=False)
state_tas.to_csv(state_tas_csv, index=False)
state_summer_tas.to_csv(state_summer_tas_csv, index=False)
state_monthly_tas.to_csv(state_monthly_tas_csv, index=False)

print("Saved:")
print(county_tas_csv)
print(state_tas_csv)
print(state_summer_tas_csv)
print(state_monthly_tas_csv)

## 6. Build extreme hot day files from daily `tasmax`

This section uses `tasmax` from CMIP6 daily data (`day`).

It creates:

- annual maximum daily max temperature
- annual number of days above 30°C and 35°C
- summer-only number of days above 30°C and 35°C
- monthly number of days above 30°C and 35°C

This cell can take longer because `tasmax` is daily data.

In [ ]:
county_heat_parts = []
county_summer_heat_parts = []
state_monthly_heat_parts = []

for scenario in SCENARIOS:
    print("\n==============================")
    print("Processing daily tasmax:", scenario)
    print("==============================")

    tasmax, lat_name, lon_name = open_cmip6_variable(
        scenario=scenario,
        variable_id="tasmax",
        table_id="day",
    )

    sampled_tasmax = sample_county_timeseries(tasmax, lat_name, lon_name, county_meta)
    sampled_tasmax_c = to_celsius(sampled_tasmax)
    sampled_tasmax_c.name = "tasmax_c"

    # -----------------------
    # Annual max of daily maximum temperature
    # -----------------------
    annual_max_tasmax_c = sampled_tasmax_c.resample(time="YS").max()
    annual_max_df = annual_da_to_county_df(
        annual_da=annual_max_tasmax_c,
        value_name="annual_max_tasmax_c",
        scenario=scenario,
    )

    # -----------------------
    # Annual hot day counts
    # -----------------------
    annual_hot_count_dfs = []
    for threshold in HOT_THRESHOLDS_C:
        value_name = f"hot_days_{threshold}c"
        print(f"  Annual count: days above {threshold}°C")
        hot_days = (sampled_tasmax_c > threshold).resample(time="YS").sum()
        hot_df = annual_da_to_county_df(
            annual_da=hot_days,
            value_name=value_name,
            scenario=scenario,
        )
        annual_hot_count_dfs.append(hot_df)

    annual_combined = annual_max_df.copy()
    for hot_df in annual_hot_count_dfs:
        annual_combined = annual_combined.merge(
            hot_df,
            on=["GEOID", "year", "scenario", "scenario_label"],
            how="left",
        )
    county_heat_parts.append(annual_combined)

    # -----------------------
    # Summer-only max and hot day counts
    # -----------------------
    summer_tasmax_c = sampled_tasmax_c.where(
        sampled_tasmax_c["time"].dt.month.isin(SUMMER_MONTHS),
        drop=True,
    )

    summer_max_tasmax_c = summer_tasmax_c.resample(time="YS").max()
    summer_max_df = annual_da_to_county_df(
        annual_da=summer_max_tasmax_c,
        value_name="summer_max_tasmax_c",
        scenario=scenario,
    )
    summer_max_df["season"] = SUMMER_LABEL

    summer_hot_count_dfs = []
    for threshold in HOT_THRESHOLDS_C:
        value_name = f"summer_hot_days_{threshold}c"
        print(f"  Summer count: days above {threshold}°C")
        summer_hot_days = (summer_tasmax_c > threshold).resample(time="YS").sum()
        summer_hot_df = annual_da_to_county_df(
            annual_da=summer_hot_days,
            value_name=value_name,
            scenario=scenario,
        )
        summer_hot_df["season"] = SUMMER_LABEL
        summer_hot_count_dfs.append(summer_hot_df)

    summer_combined = summer_max_df.copy()
    for hot_df in summer_hot_count_dfs:
        summer_combined = summer_combined.merge(
            hot_df,
            on=["GEOID", "year", "season", "scenario", "scenario_label"],
            how="left",
        )
    county_summer_heat_parts.append(summer_combined)

    # -----------------------
    # Monthly max and hot day counts, state only
    # -----------------------
    monthly_max_tasmax_c = sampled_tasmax_c.resample(time="MS").max()
    county_monthly_heat = monthly_da_to_county_df(
        monthly_da=monthly_max_tasmax_c,
        value_name="monthly_max_tasmax_c",
        scenario=scenario,
    )

    for threshold in HOT_THRESHOLDS_C:
        value_name = f"monthly_hot_days_{threshold}c"
        print(f"  Monthly count: days above {threshold}°C")
        monthly_hot_days = (sampled_tasmax_c > threshold).resample(time="MS").sum()
        monthly_hot_df = monthly_da_to_county_df(
            monthly_da=monthly_hot_days,
            value_name=value_name,
            scenario=scenario,
        )
        county_monthly_heat = county_monthly_heat.merge(
            monthly_hot_df,
            on=["GEOID", "year", "month", "scenario", "scenario_label"],
            how="left",
        )

    county_monthly_heat = county_monthly_heat.merge(county_attrs, on="GEOID", how="left")
    state_scenario_monthly_heat = aggregate_county_to_state(
        county_monthly_heat,
        value_cols=["monthly_max_tasmax_c"] + [f"monthly_hot_days_{t}c" for t in HOT_THRESHOLDS_C],
        extra_group_cols=["year", "month"],
    )
    state_monthly_heat_parts.append(state_scenario_monthly_heat)

    # Free memory before next scenario.
    del tasmax, sampled_tasmax, sampled_tasmax_c
    del annual_max_tasmax_c, annual_max_df, annual_hot_count_dfs, annual_combined
    del summer_tasmax_c, summer_max_tasmax_c, summer_max_df, summer_hot_count_dfs, summer_combined
    del monthly_max_tasmax_c, county_monthly_heat, state_scenario_monthly_heat
    gc.collect()

# -----------------------
# Annual heat: county + state
# -----------------------
county_heat = pd.concat(county_heat_parts, ignore_index=True)
county_heat = county_heat.merge(county_attrs, on="GEOID", how="left")

for threshold in HOT_THRESHOLDS_C:
    county_heat = add_annual_baseline_change(
        county_heat,
        id_cols=["GEOID", "scenario"],
        value_col=f"hot_days_{threshold}c",
    )

heat_cols = [
    "GEOID", "county", "state", "state_abbr", "ALAND",
    "year", "scenario", "scenario_label",
    "annual_max_tasmax_c",
]

for threshold in HOT_THRESHOLDS_C:
    heat_cols += [
        f"hot_days_{threshold}c",
        f"hot_days_{threshold}c_2020",
        f"hot_days_{threshold}c_change_from_2020",
    ]

county_heat = county_heat[heat_cols].sort_values(["state", "county", "scenario", "year"])

state_heat_value_cols = ["annual_max_tasmax_c"]
for threshold in HOT_THRESHOLDS_C:
    state_heat_value_cols += [
        f"hot_days_{threshold}c",
        f"hot_days_{threshold}c_2020",
        f"hot_days_{threshold}c_change_from_2020",
    ]

state_heat = aggregate_county_to_state(
    county_heat,
    value_cols=state_heat_value_cols,
    extra_group_cols=["year"],
)

# -----------------------
# Summer heat: county + state
# -----------------------
county_summer_heat = pd.concat(county_summer_heat_parts, ignore_index=True)
county_summer_heat = county_summer_heat.merge(county_attrs, on="GEOID", how="left")

for threshold in HOT_THRESHOLDS_C:
    county_summer_heat = add_annual_baseline_change(
        county_summer_heat,
        id_cols=["GEOID", "scenario"],
        value_col=f"summer_hot_days_{threshold}c",
    )

summer_heat_cols = [
    "GEOID", "county", "state", "state_abbr", "ALAND",
    "year", "season", "scenario", "scenario_label",
    "summer_max_tasmax_c",
]

for threshold in HOT_THRESHOLDS_C:
    summer_heat_cols += [
        f"summer_hot_days_{threshold}c",
        f"summer_hot_days_{threshold}c_2020",
        f"summer_hot_days_{threshold}c_change_from_2020",
    ]

county_summer_heat = county_summer_heat[summer_heat_cols].sort_values(["state", "county", "scenario", "year"])

state_summer_heat_value_cols = ["summer_max_tasmax_c"]
for threshold in HOT_THRESHOLDS_C:
    state_summer_heat_value_cols += [
        f"summer_hot_days_{threshold}c",
        f"summer_hot_days_{threshold}c_2020",
        f"summer_hot_days_{threshold}c_change_from_2020",
    ]

state_summer_heat = aggregate_county_to_state(
    county_summer_heat,
    value_cols=state_summer_heat_value_cols,
    extra_group_cols=["year", "season"],
)

# -----------------------
# Monthly heat: state only
# -----------------------
state_monthly_heat = pd.concat(state_monthly_heat_parts, ignore_index=True)

for threshold in HOT_THRESHOLDS_C:
    state_monthly_heat = add_monthly_baseline_change(
        state_monthly_heat,
        id_cols=["state", "state_abbr", "scenario"],
        value_col=f"monthly_hot_days_{threshold}c",
    )

print("county_heat rows:", len(county_heat))
print("state_heat rows:", len(state_heat))
print("state_summer_heat rows:", len(state_summer_heat))
print("state_monthly_heat rows:", len(state_monthly_heat))

display(state_heat.head())
display(state_summer_heat.head())
display(state_monthly_heat.head())

In [ ]:
county_heat_csv = OUTPUT_DIR / "county_annual_extreme_heat_cmip6_2020_2070.csv"
state_heat_csv = OUTPUT_DIR / "state_annual_extreme_heat_cmip6_2020_2070.csv"
state_summer_heat_csv = OUTPUT_DIR / "state_summer_heat_life_cmip6_2020_2070.csv"
state_monthly_heat_csv = OUTPUT_DIR / "state_monthly_heat_life_cmip6_2020_2070.csv"

county_heat.to_csv(county_heat_csv, index=False)
state_heat.to_csv(state_heat_csv, index=False)
state_summer_heat.to_csv(state_summer_heat_csv, index=False)
state_monthly_heat.to_csv(state_monthly_heat_csv, index=False)

print("Saved:")
print(county_heat_csv)
print(state_heat_csv)
print(state_summer_heat_csv)
print(state_monthly_heat_csv)

## 7. Export GeoJSON files for D3

In [ ]:
states_export = states[["STATEFP", "STUSPS", "NAME", "geometry"]].copy()
states_export = states_export.rename(columns={
    "STUSPS": "state_abbr",
    "NAME": "state",
})

counties_export = county_meta[
    ["GEOID", "STATEFP", "COUNTYFP", "county", "state", "state_abbr", "geometry"]
].copy()

states_geojson = OUTPUT_DIR / "us_states.geojson"
counties_geojson = OUTPUT_DIR / "us_counties.geojson"

states_export.to_file(states_geojson, driver="GeoJSON")
counties_export.to_file(counties_geojson, driver="GeoJSON")

print("Saved:")
print(states_geojson)
print(counties_geojson)

## 8. Create combined files for easier D3 work

These files combine average temperature and hot-day metrics into one table for each time scale.

In [ ]:
# Annual combined state file.
state_annual_combined = state_tas.merge(
    state_heat,
    on=["state", "state_abbr", "year", "scenario", "scenario_label"],
    how="left",
)

# Summer combined state file.
state_summer_combined = state_summer_tas.merge(
    state_summer_heat,
    on=["state", "state_abbr", "year", "season", "scenario", "scenario_label"],
    how="left",
)

# Monthly combined state file.
state_monthly_combined = state_monthly_tas.merge(
    state_monthly_heat,
    on=["state", "state_abbr", "year", "month", "scenario", "scenario_label"],
    how="left",
)

state_annual_combined_csv = OUTPUT_DIR / "state_annual_heat_life_combined_cmip6_2020_2070.csv"
state_summer_combined_csv = OUTPUT_DIR / "state_summer_heat_life_combined_cmip6_2020_2070.csv"
state_monthly_combined_csv = OUTPUT_DIR / "state_monthly_heat_life_combined_cmip6_2020_2070.csv"

state_annual_combined.to_csv(state_annual_combined_csv, index=False)
state_summer_combined.to_csv(state_summer_combined_csv, index=False)
state_monthly_combined.to_csv(state_monthly_combined_csv, index=False)

print("Saved combined files:")
print(state_annual_combined_csv)
print(state_summer_combined_csv)
print(state_monthly_combined_csv)

display(state_annual_combined.head())
display(state_summer_combined.head())
display(state_monthly_combined.head())

## 9. Quick validation checks

Use this section to check whether the new story direction works before updating the webpage.

In [ ]:
print("Files in output directory:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.name, "--", round(path.stat().st_size / 1_000_000, 2), "MB")

print("\nMissing values check:")
for name, df in [
    ("state_tas", state_tas),
    ("state_heat", state_heat),
    ("state_summer_combined", state_summer_combined),
    ("state_monthly_combined", state_monthly_combined),
]:
    print("\n", name)
    print(df.isna().sum().sort_values(ascending=False).head(12))

print("\nYear ranges:")
for name, df in [
    ("state_tas", state_tas),
    ("state_heat", state_heat),
    ("state_summer_combined", state_summer_combined),
    ("state_monthly_combined", state_monthly_combined),
]:
    print(name, df["year"].min(), df["year"].max(), "rows:", len(df))

print("\nExample: top states by additional 35°C+ annual hot days in 2070 under SSP585")
display(
    state_annual_combined[
        (state_annual_combined["year"] == 2070) &
        (state_annual_combined["scenario"] == "ssp585")
    ][
        ["state", "tas_change_from_2020_c", "hot_days_35c", "hot_days_35c_change_from_2020"]
    ]
    .sort_values("hot_days_35c_change_from_2020", ascending=False)
    .head(10)
)

print("\nExample: top states by additional summer 35°C+ hot days in 2070 under SSP585")
display(
    state_summer_combined[
        (state_summer_combined["year"] == 2070) &
        (state_summer_combined["scenario"] == "ssp585")
    ][
        ["state", "summer_tas_c_change_from_2020", "summer_hot_days_35c", "summer_hot_days_35c_change_from_2020"]
    ]
    .sort_values("summer_hot_days_35c_change_from_2020", ascending=False)
    .head(10)
)

print("\nExample monthly pattern: California, SSP585, 2070")
display(
    state_monthly_combined[
        (state_monthly_combined["state"] == "California") &
        (state_monthly_combined["scenario"] == "ssp585") &
        (state_monthly_combined["year"] == 2070)
    ][
        ["state", "year", "month", "monthly_tas_c", "monthly_hot_days_35c", "monthly_hot_days_35c_change_from_2020_same_month"]
    ]
)

## 10. Zip and download all outputs

Run this after the validation looks reasonable.

In [ ]:
zip_path = shutil.make_archive(
    base_name=str(OUTPUT_DIR),
    format="zip",
    root_dir=str(OUTPUT_DIR),
)

print("Created zip:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("If this is not running in Colab, download manually from:", zip_path)
    print(e)

## 11. Which files should go into the website `data/` folder?

For the current D3 map, keep using:

```text
us_states.geojson
state_annual_tas_cmip6_2020_2070.csv
state_annual_extreme_heat_cmip6_2020_2070.csv
```

For the updated daily-life story, also add:

```text
state_annual_heat_life_combined_cmip6_2020_2070.csv
state_summer_heat_life_combined_cmip6_2020_2070.csv
state_monthly_heat_life_combined_cmip6_2020_2070.csv
```

The monthly file is the most useful for a selected-state chart like:

> If this state has many 35°C+ days in July and August, what changes in daily life?